# 00. Setup and data

Run this once on a new computer, or whenever something will not import.

1. Finds the project folder.
2. Checks that the Python packages in `requirements.txt` are installed, and installs them if you ask.
3. Lists the IBM Quantum accounts saved on this computer (names only, never the API key).
4. Downloads the two cohorts (about 880 MB) into `data/`. Files already there are skipped.

In [1]:
import os, sys
from pathlib import Path
# Find the project folder (the one holding the pqk100 package), wherever Jupyter or VS Code started
def _find_root():
    starts = [Path(p).parent for p in [globals().get("__vsc_ipynb_file__")] if p]
    starts += [Path.cwd(), Path(r"D:\Active Research Projects\PQK100")]
    for s in starts:
        for p in [s, *s.parents]:
            if (p / "pqk100" / "__init__.py").exists() and (p / "credits").is_dir():
                return p.resolve()
    raise RuntimeError(f"Cannot find the PQK100 project folder from {Path.cwd()}. "
                       "Set ROOT = Path(r'D:\\Active Research Projects\\PQK100') by hand in this cell.")
ROOT = _find_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project folder:", ROOT)
import platform
print("python:", platform.python_version(), "at", sys.executable)

project folder: D:\Active Research Projects\PQK100
python: 3.14.7 at d:\Active Research Projects\PQK100\.venv\Scripts\python.exe


## 0.1 Packages

If anything is missing or has the wrong version, set `INSTALL = True` and run the cell again.
It installs into the Python that runs this notebook. Restart the kernel afterwards.

In [2]:
INSTALL = False

import importlib.metadata as md, re, subprocess
rows, bad = [], []
for line in (ROOT / "requirements.txt").read_text().split():
    name, _, want = line.partition("==")
    try:
        have = md.version(name)
    except md.PackageNotFoundError:
        have = None
    ok = have is not None and (not want or have == want)
    rows.append(f"{'ok     ' if ok else 'PROBLEM'}  {name:20s} installed {have or '-':12s} wanted {want or 'any'}")
    if not ok:
        bad.append(line)
print("\n".join(rows))
if bad and INSTALL:
    r = subprocess.run([sys.executable, "-m", "pip", "install", *bad], capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    print("Done. Restart the kernel, then run this notebook again.")
elif bad:
    print(f"\n{len(bad)} package(s) need attention. Set INSTALL = True and run this cell again.")
else:
    print("\nAll packages are in place.")

ok       qiskit               installed 2.5.1        wanted 2.5.1
ok       qiskit-aer           installed 0.17.2       wanted 0.17.2
ok       qiskit-ibm-runtime   installed 0.48.0       wanted 0.48.0
ok       numpy                installed 2.5.4        wanted any
ok       scipy                installed 1.18.1       wanted any
ok       pandas               installed 3.0.6        wanted any
ok       scikit-learn         installed 1.9.1        wanted any
ok       networkx             installed 3.7          wanted any
ok       matplotlib           installed 3.11.2       wanted any
ok       joblib               installed 1.6.0        wanted any
ok       threadpoolctl        installed 3.7.0        wanted any
ok       jupyter              installed 1.1.1        wanted any

All packages are in place.


## 0.2 Saved IBM Quantum accounts

Notebooks 04 and C3 use the account saved as `Faryad-free-ibm`.

In [3]:
ACCOUNT = "Faryad-free-ibm"
try:
    from qiskit_ibm_runtime import QiskitRuntimeService
    saved = QiskitRuntimeService.saved_accounts()
    for name, acc in saved.items():
        print(f"{name:25s} channel: {acc.get('channel', '?')}")
    print("\n" + (f"{ACCOUNT} is saved. Hardware notebooks will use it." if ACCOUNT in saved else
                  f"{ACCOUNT} is NOT saved on this computer. Save it once with\n"
                  f"QiskitRuntimeService.save_account(token='<API key>', name='{ACCOUNT}', channel='ibm_quantum_platform')"))
except ImportError:
    print("qiskit_ibm_runtime is not installed yet; see 0.1")

Faryad-free-ibm           channel: ibm_cloud
Your-free-ibm             channel: ibm_cloud

Faryad-free-ibm is saved. Hardware notebooks will use it.


## 0.3 Data

TCGA-BRCA and METABRIC from the cBioPortal datahub. If the connection drops, run the cell again; it resumes.

In [4]:
GET_DATA = True

import urllib.request
B = "https://media.githubusercontent.com/media/cBioPortal/datahub/master/public"
R = "https://raw.githubusercontent.com/cBioPortal/datahub/master/public"
FILES = [
    (f"{B}/brca_tcga_pan_can_atlas_2018/data_mrna_seq_v2_rsem.txt", "data/tcga_brca/expr.txt"),
    (f"{B}/brca_tcga_pan_can_atlas_2018/data_clinical_patient.txt", "data/tcga_brca/clin_patient.txt"),
    (f"{B}/brca_tcga_pub2015/data_clinical_sample.txt", "data/tcga_brca/clin_sample_pub2015.txt"),
    (f"{B}/brca_tcga_pub2015/data_mutations.txt", "data/tcga_brca/mutations_pub2015.txt"),
    (f"{R}/brca_tcga_pub2015/case_lists/cases_sequenced.txt", "data/tcga_brca/cases_sequenced_pub2015.txt"),
    (f"{B}/brca_metabric/data_mrna_illumina_microarray.txt", "data/metabric/expr.txt"),
    (f"{B}/brca_metabric/data_clinical_patient.txt", "data/metabric/clin_patient.txt"),
    (f"{B}/brca_metabric/data_mutations.txt", "data/metabric/mutations.txt"),
    (f"{R}/brca_metabric/case_lists/cases_sequenced.txt", "data/metabric/cases_sequenced.txt"),
]

def fetch(url, dest):
    """Download to dest. An interrupted download leaves dest.part and resumes next time."""
    dest = ROOT / dest
    if dest.exists() and dest.stat().st_size > 0:
        print(f"have  {dest.relative_to(ROOT)}  ({dest.stat().st_size / 1e6:.0f} MB)")
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_name(dest.name + ".part")
    done = part.stat().st_size if part.exists() else 0
    req = urllib.request.Request(url, headers={"Range": f"bytes={done}-"} if done else {})
    with urllib.request.urlopen(req, timeout=60) as r:
        if done and r.status != 206:
            done = 0
        total = done + int(r.headers.get("Content-Length") or 0)
        with open(part, "ab" if done else "wb") as f:
            while chunk := r.read(1 << 20):
                f.write(chunk); done += len(chunk)
    if done != total:
        raise RuntimeError(f"incomplete download of {dest.name}: run this cell again to resume")
    part.replace(dest)
    print(f"ok    {dest.relative_to(ROOT)}  ({done / 1e6:.0f} MB)")

if GET_DATA:
    for url, dest in FILES:
        fetch(url, dest)

have  data\tcga_brca\expr.txt  (155 MB)
have  data\tcga_brca\clin_patient.txt  (0 MB)
have  data\tcga_brca\clin_sample_pub2015.txt  (1 MB)
have  data\tcga_brca\mutations_pub2015.txt  (35 MB)
have  data\tcga_brca\cases_sequenced_pub2015.txt  (0 MB)
have  data\metabric\expr.txt  (689 MB)
have  data\metabric\clin_patient.txt  (0 MB)
have  data\metabric\mutations.txt  (4 MB)
have  data\metabric\cases_sequenced.txt  (0 MB)


## 0.4 Quick check

In [5]:
from pqk100.data import load_cohorts
tcga, meta = load_cohorts()          # the first call builds cache/cohorts.npz, about a minute
print(f"TCGA-BRCA {tcga.X.shape}, METABRIC {meta.X.shape}. Ready.")

TCGA-BRCA (1082, 17139), METABRIC (1980, 17139). Ready.
